# Customer Churn Prediction - Final Solution
## Playground Series S6E3

---

| Metric | Value |
|--------|-------|
| Problem Type | Binary Classification |
| Target | Churn (Yes/No) |
| Evaluation Metric | ROC-AUC |
| Current Score (V1) | 0.91404 |
| Target Score | 0.916-0.918 |
| Top Score | 0.91761 |

---

**Strategy:** Conservative + Optimized for Speed

**Key Features:**
- Original Telco data integration (auto-download)
- 100+ engineered features
- XGBoost + LightGBM + CatBoost ensemble
- Multi-seed training (3 seeds)
- Rank averaging ensemble
- 15 Optuna trials per model

**Runtime:** MacBook ~3-4h, Kaggle ~2-2.5h

---

**Author:** Pratik | **Version:** 3.0 | **Date:** 2026-03-28


## SECTION 1: Environment Setup

Auto-detects Kaggle vs MacBook and sets paths accordingly.


In [1]:
# Auto-detect environment
import os, gc, sys, platform

ON_KAGGLE = os.path.exists('/kaggle/input')
ON_MAC = platform.system() == 'Darwin'

print(f"Platform: {platform.system()}, On Kaggle: {ON_KAGGLE}, Python: {sys.version.split()[0]}")

if ON_KAGGLE:
    DATA_DIR = '/kaggle/input/competitions/playground-series-s6e3'
    OUTPUT_DIR = '/kaggle/working'
    possible_origins = ['/kaggle/input/telco-customer-churn', '/kaggle/input/ibm-telco-customer-churn']
    ORIGINAL_DIR = next((p for p in possible_origins if os.path.exists(p)), None)
    print(f"KAGGLE: Data={DATA_DIR}, Original={ORIGINAL_DIR}, Output={OUTPUT_DIR}")
else:
    DATA_DIR = './data'
    ORIGINAL_DIR = './original_data'
    OUTPUT_DIR = './output'
    print(f"MACBOOK: Data={DATA_DIR}, Original={ORIGINAL_DIR}, Output={OUTPUT_DIR}")

os.makedirs(OUTPUT_DIR, exist_ok=True)


Platform: Darwin, On Kaggle: False, Python: 3.14.3
MACBOOK: Data=./data, Original=./original_data, Output=./output


## SECTION 2: Imports


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import LabelEncoder
from scipy.stats import rankdata
from scipy.optimize import minimize
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

SEED = 42
N_FOLDS = 5
SEEDS = [42, 123, 2024]
N_TRIALS = 15

print(f"XGB:{xgb.__version__}, LGB:{lgb.__version__}, Optuna:{optuna.__version__}")
print(f"Config: Seeds={SEEDS}, Folds={N_FOLDS}, Trials={N_TRIALS}")


XGB:3.2.0, LGB:4.6.0, Optuna:4.8.0
Config: Seeds=[42, 123, 2024], Folds=5, Trials=15


venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## SECTION 3: Load Data


In [3]:
print("Loading data...")
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0}).fillna(train['Churn']).astype(int)
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Churn rate: {train['Churn'].mean()*100:.2f}%")


Loading data...
Train: (594194, 21), Test: (254655, 20)
Churn rate: 22.52%


## SECTION 4: Original Telco Data

Downloads IBM Telco Customer Churn dataset from GitHub.


In [4]:
import urllib.request
ORIGINAL_FILENAME = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
os.makedirs(ORIGINAL_DIR, exist_ok=True)
original_filepath = os.path.join(ORIGINAL_DIR, ORIGINAL_FILENAME)

if not os.path.exists(original_filepath):
    print("Downloading original Telco data...")
    url = f"https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/{ORIGINAL_FILENAME}"
    try:
        urllib.request.urlretrieve(url, original_filepath)
        print(f"Downloaded: {original_filepath}")
    except Exception as e:
        print(f"Download failed: {e}")
        USE_ORIGINAL = False
else:
    print(f"Found: {original_filepath}")

if os.path.exists(original_filepath):
    original = pd.read_csv(original_filepath)
    original['Churn'] = original['Churn'].map({'Yes': 1, 'No': 0}).fillna(original['Churn']).astype(int)
    col_map = {'customerID':'id','SeniorCitizen':'SeniorCitizen','Partner':'Partner','Dependents':'Dependents',
               'tenure':'tenure','PhoneService':'PhoneService','MultipleLines':'MultipleLines',
               'InternetService':'InternetService','OnlineSecurity':'OnlineSecurity','OnlineBackup':'OnlineBackup',
               'DeviceProtection':'DeviceProtection','TechSupport':'TechSupport','StreamingTV':'StreamingTV',
               'StreamingMovies':'StreamingMovies','Contract':'Contract','PaperlessBilling':'PaperlessBilling',
               'PaymentMethod':'PaymentMethod','MonthlyCharges':'MonthlyCharges','TotalCharges':'TotalCharges'}
    keep = [c for c in col_map.keys() if c in original.columns]
    original = original[keep].rename(columns=col_map)
    original['id'] = range(len(original))
    print(f"Original: {original.shape}, Churn rate: {original['Churn'].mean()*100:.2f}%")
    USE_ORIGINAL = True
else:
    USE_ORIGINAL = False
    original = None

print(f"Using original data: {USE_ORIGINAL}")


Found: ./original_data/WA_Fn-UseC_-Telco-Customer-Churn.csv


KeyError: 'Churn'

## SECTION 5: Merge Data


In [ ]:
if USE_ORIGINAL:
    common = [c for c in train.columns if c in original.columns]
    train_combined = pd.concat([train[common], original[common]], ignore_index=True)
    keys = ['gender','SeniorCitizen','Partner','Dependents','tenure','PhoneService','InternetService','Contract','MonthlyCharges','Churn']
    keys = [k for k in keys if k in train_combined.columns]
    before = len(train_combined)
    train_combined = train_combined.drop_duplicates(subset=keys, keep='first')
    print(f"Merged: {before:,} -> {len(train_combined):,} (removed {before-len(train_combined):,})")
    train_final = train_combined
else:
    train_final = train
    print("Using synthetic data only")

test_final = test
print(f"Final: Train={len(train_final):,}, Test={len(test_final):,}")


## SECTION 6: Feature Engineering

100+ features including tenure, charges, services, risk flags.


In [ ]:
def fe(df, is_train=True, stats=None):
    df = df.copy()
    # Tenure
    df['tenure_bucket'] = pd.cut(df['tenure'], bins=[-1,6,12,24,48,72,999], labels=[0,1,2,3,4,5]).astype(int)
    df['is_new'] = (df['tenure']<=6).astype(int)
    df['is_long_term'] = (df['Contract']!='Month-to-month').astype(int)
    df['tenure_norm'] = df['tenure']/72
    df['tenure_sq'] = df['tenure']**2
    df['tenure_log'] = np.log1p(df['tenure'])
    
    # Charges
    df['charge_per_tenure'] = df['MonthlyCharges']/(df['tenure']+1)
    df['total_monthly_ratio'] = df['TotalCharges']/(df['MonthlyCharges']+1)
    df['exp_total'] = df['MonthlyCharges']*df['tenure']
    df['charge_dev'] = df['TotalCharges']-df['exp_total']
    df['log_monthly'] = np.log1p(df['MonthlyCharges'])
    df['log_total'] = np.log1p(df['TotalCharges'].fillna(0))
    
    if is_train:
        cs = df.groupby('Contract')['MonthlyCharges'].agg(['mean','std','median']).reset_index()
        cs.columns = ['Contract','cm','cs','cmed']
        stats = {'cs': cs}
    if stats:
        df = df.merge(stats['cs'], on='Contract', how='left')
        df['charge_z'] = (df['MonthlyCharges']-df['cm'])/(df['cs']+1e-6)
        df.drop(['cm','cs','cmed'], axis=1, inplace=True)
    
    # Services
    for c in ['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']:
        if c in df.columns:
            df[f'{c}_flag'] = (df[c]=='Yes').astype(int) if df[c].dtype=='object' else 0
    flags = [f'{c}_flag' for c in ['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies'] if f'{c}_flag' in df.columns]
    df['n_services'] = df[flags].sum(axis=1)
    df['all_services'] = (df['n_services']==6).astype(int)
    df['sec_bundle'] = ((df.get('OnlineSecurity_flag',0)==1)&(df.get('TechSupport_flag',0)==1)).astype(int)
    df['stream_bundle'] = ((df.get('StreamingTV_flag',0)==1)&(df.get('StreamingMovies_flag',0)==1)).astype(int)
    
    # Risk flags
    cm = df['MonthlyCharges'].median() if is_train else 70
    df['high_risk'] = ((df['Contract']=='Month-to-month')&(df['tenure']<=12)&(df['MonthlyCharges']>cm)).astype(int)
    df['elec_check'] = ((df['Contract']=='Month-to-month')&(df['PaymentMethod']=='Electronic check')).astype(int)
    df['fiber_high'] = ((df['InternetService']=='Fiber optic')&(df['MonthlyCharges']>cm)).astype(int)
    df['no_sec_high'] = ((df.get('OnlineSecurity_flag',0)==0)&(df['MonthlyCharges']>cm)).astype(int)
    df['new_fiber_no_sec'] = ((df['tenure']<=6)&(df['InternetService']=='Fiber optic')&(df.get('OnlineSecurity_flag',0)==0)).astype(int)
    
    # Family
    if 'Partner' in df.columns:
        df['has_partner'] = (df['Partner']=='Yes').astype(int) if df['Partner'].dtype=='object' else 0
    if 'Dependents' in df.columns:
        df['has_dep'] = (df['Dependents']=='Yes').astype(int) if df['Dependents'].dtype=='object' else 0
    df['has_family'] = ((df.get('has_partner',0)==1)|(df.get('has_dep',0)==1)).astype(int)
    
    # Interactions
    df['tenure_x_contract'] = df['tenure_bucket']*df['is_long_term']
    df['tenure_x_services'] = df['tenure']*df['n_services']
    df['tenure_x_monthly'] = df['tenure']*df['MonthlyCharges']
    
    # Combos
    df['Contract_Internet'] = df['Contract'].astype(str)+'_'+df['InternetService'].astype(str)
    df['Contract_Payment'] = df['Contract'].astype(str)+'_'+df['PaymentMethod'].astype(str)
    df['Internet_Payment'] = df['InternetService'].astype(str)+'_'+df['PaymentMethod'].astype(str)
    
    # Payment
    df['is_elec'] = (df['PaymentMethod']=='Electronic check').astype(int)
    df['is_auto'] = df['PaymentMethod'].str.contains('auto', case=False, na=False).astype(int)
    df['is_paperless'] = (df['PaperlessBilling']=='Yes').astype(int) if 'PaperlessBilling' in df.columns and df['PaperlessBilling'].dtype=='object' else 0
    
    return df, stats

print("Feature engineering...")
train_fe, stats = fe(train_final, is_train=True)
test_fe, _ = fe(test_final, is_train=False, stats=stats)
print(f"Features: {len(train_fe.columns)}")


## SECTION 7: Encoding

Target encoding, frequency encoding, label encoding.


In [ ]:
# Target encoding
cat_cols = [c for c in train_fe.select_dtypes(include='object').columns if c != 'id']
for col in cat_cols:
    gm = train_fe['Churn'].mean()
    train_fe[f'{col}_te'] = 0.0
    for tr, va in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(train_fe, train_fe['Churn']):
        means = train_fe.iloc[tr].groupby(col)['Churn'].mean()
        train_fe.loc[va, f'{col}_te'] = train_fe.iloc[va][col].map(means)
    train_fe[f'{col}_te'] = train_fe[f'{col}_te'].fillna(gm)
    test_fe[f'{col}_te'] = test_fe[col].map(train_fe.groupby(col)['Churn'].mean()).fillna(gm)

# Frequency encoding
for col in cat_cols:
    fm = train_fe[col].value_counts(normalize=True).to_dict()
    train_fe[f'{col}_freq'] = train_fe[col].map(fm)
    test_fe[f'{col}_freq'] = test_fe[col].map(fm).fillna(0)

# Label encoding
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train_fe[col].astype(str), test_fe[col].astype(str)])
    le.fit(combined)
    train_fe[col] = le.transform(train_fe[col].astype(str))
    test_fe[col] = le.transform(test_fe[col].astype(str))

drop = ['id', 'Churn']
feat_cols = [c for c in train_fe.columns if c not in drop]
X = train_fe[feat_cols].values.astype(np.float32)
y = train_fe['Churn'].values
X_test = test_fe[feat_cols].values.astype(np.float32)
test_ids = test_fe['id'].values

print(f"Features: {len(feat_cols)}, Train: {len(X):,}, Test: {len(X_test):,}")


## SECTION 8: Hyperparameter Tuning

Optuna with 15 trials per model.


In [ ]:
print("="*60)
print("XGBOOST TUNING")
print("="*60)

def obj_xgb(t):
    p = {'n_estimators':t.suggest_int('n',500,2000),'max_depth':t.suggest_int('d',4,7),
         'learning_rate':t.suggest_float('lr',0.01,0.15,log=True),'subsample':t.suggest_float('sub',0.7,0.9),
         'colsample_bytree':t.suggest_float('col',0.5,0.8),'min_child_weight':t.suggest_int('mcw',10,50),
         'reg_alpha':t.suggest_float('ra',0.01,10,log=True),'reg_lambda':t.suggest_float('rl',0.1,10,log=True),
         'gamma':t.suggest_float('g',0.5,5),'scale_pos_weight':t.suggest_float('spw',3,4),
         'random_state':SEED,'eval_metric':'auc','tree_method':'hist'}
    sc = []
    for tr,va in StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED).split(X,y):
        m = xgb.XGBClassifier(**p)
        m.fit(X[tr],y[tr],eval_set=[(X[va],y[va])],verbose=0)
        sc.append(roc_auc_score(y[va],m.predict_proba(X[va])[:,1]))
    return np.mean(sc)

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(obj_xgb, n_trials=N_TRIALS, show_progress_bar=True)
print(f"Best: {study_xgb.best_value:.6f}")


In [ ]:
print("="*60)
print("LIGHTGBM TUNING")
print("="*60)

def obj_lgb(t):
    try:
        lgb.LGBMClassifier(device='gpu',n_estimators=10).fit(X[:100],y[:100],verbose=-1)
        gpu = True
    except: gpu = False
    p = {'n_estimators':t.suggest_int('n',500,2000),'max_depth':t.suggest_int('d',4,8),
         'learning_rate':t.suggest_float('lr',0.01,0.15,log=True),'num_leaves':t.suggest_int('nl',30,100),
         'subsample':t.suggest_float('sub',0.7,0.9),'colsample_bytree':t.suggest_float('col',0.5,0.8),
         'min_child_samples':t.suggest_int('mcs',30,100),'reg_alpha':t.suggest_float('ra',0.01,10,log=True),
         'reg_lambda':t.suggest_float('rl',0.1,10,log=True),'random_state':SEED,'verbose':-1}
    if gpu: p['device']='gpu'
    sc = []
    for tr,va in StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED).split(X,y):
        m = lgb.LGBMClassifier(**p)
        m.fit(X[tr],y[tr],eval_set=[(X[va],y[va])],callbacks=[lgb.log_evaluation(0)])
        sc.append(roc_auc_score(y[va],m.predict_proba(X[va])[:,1]))
    return np.mean(sc)

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(obj_lgb, n_trials=N_TRIALS, show_progress_bar=True)
print(f"Best: {study_lgb.best_value:.6f}")


In [ ]:
print("="*60)
print("CATBOOST TUNING")
print("="*60)

def obj_cat(t):
    try:
        CatBoostClassifier(task_type='GPU',iterations=10,verbose=0).fit(X[:100],y[:100])
        gpu = True
    except: gpu = False
    p = {'iterations':t.suggest_int('i',500,2000),'depth':t.suggest_int('d',4,7),
         'learning_rate':t.suggest_float('lr',0.01,0.15,log=True),'l2_leaf_reg':t.suggest_float('l2',3,20,log=True),
         'min_data_in_leaf':t.suggest_int('mdil',30,150),'bootstrap_type':'MVS',
         'subsample':t.suggest_float('sub',0.7,0.9),'colsample_bylevel':t.suggest_float('col',0.5,0.8),
         'random_seed':SEED,'verbose':0,'eval_metric':'AUC'}
    if gpu: p.update({'task_type':'GPU','devices':'0'})
    else: p['task_type']='CPU'
    sc = []
    for tr,va in StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED).split(X,y):
        m = CatBoostClassifier(**p)
        m.fit(X[tr],y[tr],eval_set=(X[va],y[va]),verbose=0)
        sc.append(roc_auc_score(y[va],m.predict_proba(X[va])[:,1]))
    return np.mean(sc)

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(obj_cat, n_trials=N_TRIALS, show_progress_bar=True)
print(f"Best: {study_cat.best_value:.6f}")


## SECTION 9: Multi-Seed Training

3 seeds x 3 models x 5 folds = 45 models


In [ ]:
xgb_p = {**study_xgb.best_params,'eval_metric':'auc','tree_method':'hist'}
lgb_p = {**study_lgb.best_params,'verbose':-1}
cat_p = {**study_cat.best_params,'verbose':0,'eval_metric':'AUC'}

try:
    lgb.LGBMClassifier(device='gpu',n_estimators=10).fit(X[:100],y[:100],verbose=-1)
    lgb_p['device']='gpu'
except: pass
try:
    CatBoostClassifier(task_type='GPU',iterations=10,verbose=0).fit(X[:100],y[:100])
    cat_p.update({'task_type':'GPU','devices':'0'})
except: cat_p['task_type']='CPU'

n = len(SEEDS)
oof_xgb = np.zeros((n,len(X)))
oof_lgb = np.zeros((n,len(X)))
oof_cat = np.zeros((n,len(X)))
test_xgb = np.zeros((n,len(X_test)))
test_lgb = np.zeros((n,len(X_test)))
test_cat = np.zeros((n,len(X_test)))

for si, seed in enumerate(SEEDS):
    print(f"\nSEED {seed} ({si+1}/{n})")
    for fold, (tr,va) in enumerate(StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=seed).split(X,y)):
        xgb_m = xgb.XGBClassifier(**{**xgb_p,'random_state':seed})
        xgb_m.fit(X[tr],y[tr],eval_set=[(X[va],y[va])],verbose=0)
        oof_xgb[si,va] = xgb_m.predict_proba(X[va])[:,1]
        test_xgb[si] += xgb_m.predict_proba(X_test)[:,1]/N_FOLDS
        
        lgb_m = lgb.LGBMClassifier(**{**lgb_p,'random_state':seed})
        lgb_m.fit(X[tr],y[tr],eval_set=[(X[va],y[va])],callbacks=[lgb.log_evaluation(0)])
        oof_lgb[si,va] = lgb_m.predict_proba(X[va])[:,1]
        test_lgb[si] += lgb_m.predict_proba(X_test)[:,1]/N_FOLDS
        
        cat_m = CatBoostClassifier(**{**cat_p,'random_seed':seed})
        cat_m.fit(X[tr],y[tr],eval_set=(X[va],y[va]),verbose=0)
        oof_cat[si,va] = cat_m.predict_proba(X[va])[:,1]
        test_cat[si] += cat_m.predict_proba(X_test)[:,1]/N_FOLDS
        
        print(f"Fold {fold+1}: XGB={roc_auc_score(y[va],oof_xgb[si,va]):.6f} LGB={roc_auc_score(y[va],oof_lgb[si,va]):.6f} CAT={roc_auc_score(y[va],oof_cat[si,va]):.6f}")
        del xgb_m,lgb_m,cat_m; gc.collect()

oof_xgb_avg = oof_xgb.mean(axis=0)
oof_lgb_avg = oof_lgb.mean(axis=0)
oof_cat_avg = oof_cat.mean(axis=0)
test_xgb_avg = test_xgb.mean(axis=0)
test_lgb_avg = test_lgb.mean(axis=0)
test_cat_avg = test_cat.mean(axis=0)

print(f"\nOOF: XGB={roc_auc_score(y,oof_xgb_avg):.6f} LGB={roc_auc_score(y,oof_lgb_avg):.6f} CAT={roc_auc_score(y,oof_cat_avg):.6f}")


## SECTION 10: Ensemble

Rank averaging.


In [ ]:
oof_list = [oof_xgb_avg, oof_lgb_avg, oof_cat_avg]
test_list = [test_xgb_avg, test_lgb_avg, test_cat_avg]
names = ['XGBoost', 'LightGBM', 'CatBoost']

oof_simple = np.mean(oof_list, axis=0)
test_simple = np.mean(test_list, axis=0)
print(f"Simple: {roc_auc_score(y,oof_simple):.6f}")

ranks = [rankdata(p) for p in oof_list]
oof_rank = np.mean(ranks, axis=0)/len(oof_rank)
test_rank = np.mean([rankdata(p) for p in test_list], axis=0)/len(X_test)
print(f"Rank: {roc_auc_score(y,oof_rank):.6f}")

def opt_w(oof_list, y):
    def obj(w):
        w = np.abs(w)/np.sum(np.abs(w))
        return -roc_auc_score(y, sum(wi*o for wi,o in zip(w,oof_list)))
    return np.abs(minimize(obj, np.ones(len(oof_list))/len(oof_list), method='Nelder-Mead').x)/np.sum(np.abs(minimize(obj, np.ones(len(oof_list))/len(oof_list), method='Nelder-Mead').x))

ow = opt_w(oof_list, y)
oof_w = sum(wi*o for wi,o in zip(ow,oof_list))
test_w = sum(wi*t for wi,t in zip(ow,test_list))
print(f"Weighted: {roc_auc_score(y,oof_w):.6f}")
for n,w in zip(names,ow): print(f"  {n}: {w:.4f}")

best = max([('Simple',roc_auc_score(y,oof_simple),test_simple),('Rank',roc_auc_score(y,oof_rank),test_rank),('Weighted',roc_auc_score(y,oof_w),test_w)], key=lambda x:x[1])
print(f"\nSELECTED: {best[0]} ({best[1]:.6f})")
best_preds = best[2]


## SECTION 11: Visualizations


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
scores = [roc_auc_score(y,oof_xgb_avg), roc_auc_score(y,oof_lgb_avg), roc_auc_score(y,oof_cat_avg), best[1]]
axes[0].bar(['XGB','LGB','CAT','Ensemble'], scores, color=['#42a5f5','#66bb6a','#ffa726','#ff6b6b'], edgecolor='black')
axes[0].set_ylabel('ROC-AUC'); axes[0].set_title('Model Comparison'); axes[0].set_ylim(min(scores)-0.005,max(scores)+0.005)
for i,s in enumerate(scores): axes[0].text(i,s+0.001,f'{s:.5f}',ha='center')

for oof, name, c in [(oof_xgb_avg,'XGB','#42a5f5'),(oof_lgb_avg,'LGB','#66bb6a'),(oof_cat_avg,'CAT','#ffa726'),(oof_simple,'Avg','#ff6b6b')]:
    fpr,tpr,_ = roc_curve(y,oof)
    axes[1].plot(fpr,tpr,label=f'{name} ({roc_auc_score(y,oof):.5f})',color=c,lw=2)
axes[1].plot([0,1],[0,1],'k--',alpha=0.5); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_comparison.png', dpi=150); plt.show()


In [ ]:
plt.figure(figsize=(10,4))
plt.hist(best_preds, bins=50, color='coral', edgecolor='black', alpha=0.7)
plt.xlabel('Predicted Churn Probability'); plt.ylabel('Count'); plt.title('Prediction Distribution')
plt.savefig(f'{OUTPUT_DIR}/prediction_distribution.png', dpi=150); plt.show()
print(f"Predictions: Min={best_preds.min():.4f}, Max={best_preds.max():.4f}, Mean={best_preds.mean():.4f}")


## SECTION 12: Generate Submission

Output: submission.csv


In [ ]:
submission = sample_sub.copy()
submission['Churn'] = best_preds

print(f"Shape: {submission.shape}, NaN: {submission['Churn'].isna().sum()}")

submission.to_csv(f'{OUTPUT_DIR}/submission.csv', index=False)
print(f"\nSAVED: {OUTPUT_DIR}/submission.csv")
print(f"Size: {os.path.getsize(f'{OUTPUT_DIR}/submission.csv')/1024:.1f} KB")

pd.DataFrame({'id':test_ids,'xgb':test_xgb_avg,'lgb':test_lgb_avg,'cat':test_cat_avg,
              'simple':test_simple,'rank':test_rank,'weighted':test_w}).to_csv(f'{OUTPUT_DIR}/all_predictions.csv', index=False)
print(f"All preds: {OUTPUT_DIR}/all_predictions.csv")


## SECTION 13: Summary

### Final Results

| Metric | Value |
|--------|-------|
| Best OOF | TBD (after running) |
| Expected LB | 0.916-0.918 |
| Target | 0.917+ (Top 10%) |

### Configuration

| Component | Value |
|-----------|-------|
| Seeds | [42, 123, 2024] |
| Folds | 5 |
| Trials | 15 per model |
| Models | XGBoost, LightGBM, CatBoost |

### Key Features
- Original Telco data auto-download
- 100+ engineered features
- Multi-seed ensemble
- Rank averaging

---

**Competition:** Playground Series S6E3
**Author:** Pratik | **Version:** 3.0 | **Date:** 2026-03-28
